# LeetCode 342 — Power of Four bằng Python

**Link đề:** https://leetcode.com/problems/power-of-four/
**Độ khó:** Easy · **Chủ đề:** Math, Bit Manipulation, Recursion

```python
class Solution:
    def isPowerOfFour(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu lũy thừa của 4 trong **hệ nhị phân**: đúng **một bit 1**, nằm ở **vị trí chẵn**.
2. Ôn lại cách chia lặp (giống Power of Three) và thấy vì sao mẹo "ước của lũy thừa lớn nhất" **không** dùng được cho 4.
3. Học mẹo bit `n & (n - 1) == 0` (lũy thừa của 2) và **mặt nạ bit** (bitmask) `0x55555555`.
4. Học mẹo toán `n % 3 == 1` để phân biệt `4^k` với `2·4^k`.
5. Trace, chứng minh, so sánh 5 cách; giải follow-up "không vòng lặp / đệ quy".

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho số nguyên `n`. Trả về `True` nếu `n` là lũy thừa của 4, ngược lại `False`.

`n` là lũy thừa của 4 nếu tồn tại số nguyên `x` sao cho `n == 4^x`.

| Input `n` | Output | Giải thích |
|---:|:---:|---|
| `16` | `True` | `4^2` |
| `5` | `False` | |
| `1` | `True` | `4^0` |
| `8` | `False` | lũy thừa của 2 nhưng **không** phải của 4 |
| `0`, `-4` | `False` | lũy thừa của 4 luôn dương |

**Constraints:** `-2^31 <= n <= 2^31 - 1`.

**Follow up:** Bạn có giải được **không dùng vòng lặp / đệ quy** không?

In [1]:
powers4 = []
p = 1
while p <= 2**31 - 1:
    powers4.append(p)
    p *= 4
print(powers4)
print("Số lượng:", len(powers4), "| lớn nhất = 4 **", len(powers4) - 1)

[1, 4, 16, 64, 256, 1024, 4096, 16384, 65536, 262144, 1048576, 4194304, 16777216, 67108864, 268435456, 1073741824]
Số lượng: 16 | lớn nhất = 4 ** 15


## 2. Nền tảng: nhìn lũy thừa của 4 trong hệ nhị phân

`4 = 2^2`, nên `4^k = 2^(2k)`: **một bit 1 duy nhất ở vị trí chẵn** (vị trí đếm từ 0 bên phải).

| n | nhị phân | vị trí bit 1 | lũy thừa của 4? |
|---:|---:|:---:|:---:|
| 1 | 1 | 0 | ✔ |
| 2 | 10 | 1 | ✘ |
| 4 | 100 | 2 | ✔ |
| 8 | 1000 | 3 | ✘ |
| 16 | 10000 | 4 | ✔ |
| 32 | 100000 | 5 | ✘ |

In [2]:
for n in [1, 2, 4, 8, 16, 32, 64]:
    pos = n.bit_length() - 1        # bit_length(): số bit cần để biểu diễn n
    print(f"{n:>3} = {n:>8b} | vị trí bit 1 = {pos} | chẵn? {pos % 2 == 0}")

  1 =        1 | vị trí bit 1 = 0 | chẵn? True
  2 =       10 | vị trí bit 1 = 1 | chẵn? False
  4 =      100 | vị trí bit 1 = 2 | chẵn? True
  8 =     1000 | vị trí bit 1 = 3 | chẵn? False
 16 =    10000 | vị trí bit 1 = 4 | chẵn? True
 32 =   100000 | vị trí bit 1 = 5 | chẵn? False
 64 =  1000000 | vị trí bit 1 = 6 | chẵn? True


### 2.1. `n & (n - 1)` — kiểm tra "chỉ có một bit 1" (lũy thừa của 2)

Trừ 1 làm bit 1 thấp nhất thành 0 và các bit sau nó thành 1. Nếu `n` chỉ có **một** bit 1, thì `n & (n - 1) == 0`.

In [3]:
for n in [8, 12, 16, 6, 1]:
    print(f"n = {n:>2} = {n:05b} | n-1 = {n - 1:05b} | n & (n-1) = {n & (n - 1):05b} -> một bit 1? {(n & (n - 1)) == 0}")

n =  8 = 01000 | n-1 = 00111 | n & (n-1) = 00000 -> một bit 1? True
n = 12 = 01100 | n-1 = 01011 | n & (n-1) = 01000 -> một bit 1? False
n = 16 = 10000 | n-1 = 01111 | n & (n-1) = 00000 -> một bit 1? True
n =  6 = 00110 | n-1 = 00101 | n & (n-1) = 00100 -> một bit 1? False
n =  1 = 00001 | n-1 = 00000 | n & (n-1) = 00000 -> một bit 1? True


### 2.2. Số thập lục phân (hex) và mặt nạ `0x55555555`

- `0x...` là cách viết số trong hệ 16. Mỗi chữ số hex = 4 bit. `0x5 = 0101₂`.
- `0x55555555` = `0101 0101 ... 0101` (32 bit): **bit 1 ở mọi vị trí chẵn** (0, 2, 4, …, 30).
- `n & mask != 0` nghĩa là `n` có ít nhất một bit 1 ở vị trí mà mask có bit 1.

In [4]:
mask = 0x55555555
print(mask)
print(f"{mask:032b}")
print(hex(255), 0xFF, 0b1111)       # cùng giá trị 255 / 255 / 15 viết bằng các hệ khác nhau
for n in [4, 8, 16, 32]:
    print(f"{n:>2}: n & mask = {n & mask} -> bit 1 ở vị trí chẵn? {(n & mask) != 0}")

1431655765
01010101010101010101010101010101
0xff 255 15
 4: n & mask = 4 -> bit 1 ở vị trí chẵn? True
 8: n & mask = 0 -> bit 1 ở vị trí chẵn? False
16: n & mask = 16 -> bit 1 ở vị trí chẵn? True
32: n & mask = 0 -> bit 1 ở vị trí chẵn? False


### 2.3. Đồng dư: `4^k % 3 == 1` nhưng `2·4^k % 3 == 2`

Vì `4 ≡ 1 (mod 3)`, nên `4^k ≡ 1^k = 1 (mod 3)`. Còn lũy thừa của 2 có số mũ lẻ: `2^(2k+1) = 2·4^k ≡ 2 (mod 3)`.

In [5]:
for e in range(9):
    print(f"2^{e} = {2 ** e:>3} | % 3 = {(2 ** e) % 3} | lũy thừa của 4? {e % 2 == 0}")

2^0 =   1 | % 3 = 1 | lũy thừa của 4? True
2^1 =   2 | % 3 = 2 | lũy thừa của 4? False
2^2 =   4 | % 3 = 1 | lũy thừa của 4? True
2^3 =   8 | % 3 = 2 | lũy thừa của 4? False
2^4 =  16 | % 3 = 1 | lũy thừa của 4? True
2^5 =  32 | % 3 = 2 | lũy thừa của 4? False
2^6 =  64 | % 3 = 1 | lũy thừa của 4? True
2^7 = 128 | % 3 = 2 | lũy thừa của 4? False
2^8 = 256 | % 3 = 1 | lũy thừa của 4? True


## 3. Cách trực giác — chia cho 4 tới khi không chia được

Giống Power of Three: chặn `n <= 0`, lặp `n //= 4` khi `n % 4 == 0`, cuối cùng kiểm tra `n == 1`.

In [6]:
def is_power_of_four_loop(n: int) -> bool:
    if n <= 0:
        return False
    while n % 4 == 0:
        n //= 4
    return n == 1

for n in [16, 5, 1, 8, 0, -4, 64, 32]:
    print(n, "->", is_power_of_four_loop(n))

16 -> True
5 -> False
1 -> True
8 -> False
0 -> False
-4 -> False
64 -> True
32 -> False


### 3.1. Cách đệ quy

In [7]:
def is_power_of_four_recursive(n: int) -> bool:
    if n == 1:
        return True
    if n <= 0 or n % 4 != 0:
        return False
    return is_power_of_four_recursive(n // 4)

print([n for n in range(-10, 300) if is_power_of_four_recursive(n)])

[1, 4, 16, 64, 256]


## 4. Nút thắt — follow-up "không vòng lặp", và mẹo của bài 326 thất bại

Với Power of Three ta dùng `3^19 % n == 0`. Thử y hệt với `4^15`:

In [8]:
MAX_POW4 = 4 ** 15
fake = [n for n in range(1, 100) if MAX_POW4 % n == 0]
print("Ước của 4^15 trong 1..99:", fake)
print("Trong đó KHÔNG phải lũy thừa của 4:", [n for n in fake if not is_power_of_four_loop(n)])

Ước của 4^15 trong 1..99: [1, 2, 4, 8, 16, 32, 64]
Trong đó KHÔNG phải lũy thừa của 4: [2, 8, 32]


Vì 4 **không** nguyên tố (`4 = 2·2`), ước của `4^15 = 2^30` là **mọi lũy thừa của 2** tới `2^30`, kể cả `2, 8, 32…`. Ta cần một ý tưởng khác để loại các lũy thừa **lẻ** của 2.

## 5. Insight — hai điều kiện

```text
n là lũy thừa của 4
⇔ n > 0
  và n là lũy thừa của 2           (n & (n - 1) == 0)
  và bit 1 duy nhất ở vị trí chẵn  (n & 0x55555555 != 0)    — hoặc  (n % 3 == 1)
```

Hai cách cho điều kiện cuối:
- **Bitmask**: bit 1 phải "trúng" một vị trí chẵn của mask.
- **Modulo 3**: `2^(chẵn) % 3 == 1`, `2^(lẻ) % 3 == 2`.

## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — số dương:** `n > 0`. (Số 0: `0 & -1 == 0` sẽ lọt qua điều kiện 2 nếu quên!)

**Đoạn 2 — lũy thừa của 2:** `(n & (n - 1)) == 0`.

**Đoạn 3 — vị trí chẵn:** `(n & 0x55555555) != 0`.

Ghép bằng `and`. Viết ngoặc đầy đủ cho rõ ràng.

In [9]:
print("0 & (0 - 1) =", 0 & (0 - 1), "-> quên n > 0 thì 0 bị coi là lũy thừa của 2!")

def is_power_of_four_mask(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and (n & 0x55555555) != 0

def is_power_of_four_mod3(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and n % 3 == 1

print([n for n in range(-10, 1100) if is_power_of_four_mask(n)])
print([n for n in range(-10, 1100) if is_power_of_four_mod3(n)])

0 & (0 - 1) = 0 -> quên n > 0 thì 0 bị coi là lũy thừa của 2!
[1, 4, 16, 64, 256, 1024]
[1, 4, 16, 64, 256, 1024]


### 6.1. Cách thứ 5 — `bit_length()` (vị trí bit chẵn ⇔ `bit_length()` lẻ)

`4^k` có `bit_length() = 2k + 1` (số lẻ).

In [10]:
def is_power_of_four_bitlen(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and n.bit_length() % 2 == 1

print([n for n in range(1, 1100) if is_power_of_four_bitlen(n)])

[1, 4, 16, 64, 256, 1024]


## 7. TRACE từng bước

### 7.1. Trace ba điều kiện của cách bitmask

Bảng tay:

| n | n > 0 | n & (n-1) | lũy thừa 2? | n & mask | vị trí chẵn? | Kết quả |
|---:|:---:|---:|:---:|---:|:---:|:---:|
| 16 | ✔ | 0 | ✔ | 16 | ✔ | True |
| 8 | ✔ | 0 | ✔ | 0 | ✘ | False |
| 12 | ✔ | 8 | ✘ | — | — | False |
| 0 | ✘ | — | — | — | — | False |

In [11]:
def trace_power_of_four(n: int) -> bool:
    mask = 0x55555555
    print(f"n = {n}" + (f" = {n:b}₂" if n > 0 else ""))
    if not n > 0:
        print("  n > 0? False -> False")
        return False
    a = n & (n - 1)
    print(f"  n & (n-1) = {a:b}₂ = {a} -> lũy thừa của 2? {a == 0}")
    if a != 0:
        return False
    m = n & mask
    print(f"  n & 0x55555555 = {m} -> bit 1 ở vị trí chẵn? {m != 0}")
    print(f"  (đối chiếu) n % 3 = {n % 3}")
    return m != 0

for n in [16, 8, 12, 0, 1, 1073741824]:
    print("=>", trace_power_of_four(n))

n = 16 = 10000₂
  n & (n-1) = 0₂ = 0 -> lũy thừa của 2? True
  n & 0x55555555 = 16 -> bit 1 ở vị trí chẵn? True
  (đối chiếu) n % 3 = 1
=> True
n = 8 = 1000₂
  n & (n-1) = 0₂ = 0 -> lũy thừa của 2? True
  n & 0x55555555 = 0 -> bit 1 ở vị trí chẵn? False
  (đối chiếu) n % 3 = 2
=> False
n = 12 = 1100₂
  n & (n-1) = 1000₂ = 8 -> lũy thừa của 2? False
=> False
n = 0
  n > 0? False -> False
=> False
n = 1 = 1₂
  n & (n-1) = 0₂ = 0 -> lũy thừa của 2? True
  n & 0x55555555 = 1 -> bit 1 ở vị trí chẵn? True
  (đối chiếu) n % 3 = 1
=> True
n = 1073741824 = 1000000000000000000000000000000₂
  n & (n-1) = 0₂ = 0 -> lũy thừa của 2? True
  n & 0x55555555 = 1073741824 -> bit 1 ở vị trí chẵn? True
  (đối chiếu) n % 3 = 1
=> True


### 7.2. Trace cách chia lặp

In [12]:
def trace_loop(n: int) -> bool:
    print(f"n = {n}")
    if n <= 0:
        print("  n <= 0 -> False")
        return False
    while n % 4 == 0:
        print(f"  {n} % 4 == 0 -> n = {n // 4}")
        n //= 4
    print(f"  dừng ở n = {n} -> {n == 1}")
    return n == 1

trace_loop(64)
trace_loop(32)

n = 64
  64 % 4 == 0 -> n = 16
  16 % 4 == 0 -> n = 4
  4 % 4 == 0 -> n = 1
  dừng ở n = 1 -> True
n = 32
  32 % 4 == 0 -> n = 8
  8 % 4 == 0 -> n = 2
  dừng ở n = 2 -> False


False

### 7.3. Bảng tổng hợp — ba cách "vị trí chẵn" luôn đồng ý với nhau

Với mọi lũy thừa của 2 từ `2^0` tới `2^12`, ta in: vị trí bit `e`, `n & mask`, `n % 3`, `bit_length()`. Quan sát: cột `e chẵn` trùng hoàn toàn với `mask ≠ 0`, `n % 3 == 1` và `bit_length() lẻ`. Đây là cách **kiểm chứng chéo** trước khi tin một mẹo.

Câu hỏi tự kiểm: vì sao chỉ cần xét lũy thừa của 2? (Vì điều kiện `n & (n - 1) == 0` đã lọc mọi số khác.)

In [13]:
print(" e |     n | e chẵn | n & mask != 0 | n % 3 | bit_length")
for e in range(13):
    n = 1 << e                      # 1 << e = 2^e: dịch bit 1 sang trái e vị trí
    print(f"{e:>2} | {n:>5} | {str(e % 2 == 0):>6} | {str((n & 0x55555555) != 0):>13} | {n % 3:>5} | {n.bit_length():>10}")
    assert (e % 2 == 0) == ((n & 0x55555555) != 0) == (n % 3 == 1) == (n.bit_length() % 2 == 1)
print("Bốn cột điều kiện khớp nhau với mọi e = 0..12")

 e |     n | e chẵn | n & mask != 0 | n % 3 | bit_length
 0 |     1 |   True |          True |     1 |          1
 1 |     2 |  False |         False |     2 |          2
 2 |     4 |   True |          True |     1 |          3
 3 |     8 |  False |         False |     2 |          4
 4 |    16 |   True |          True |     1 |          5
 5 |    32 |  False |         False |     2 |          6
 6 |    64 |   True |          True |     1 |          7
 7 |   128 |  False |         False |     2 |          8
 8 |   256 |   True |          True |     1 |          9
 9 |   512 |  False |         False |     2 |         10
10 |  1024 |   True |          True |     1 |         11
11 |  2048 |  False |         False |     2 |         12
12 |  4096 |   True |          True |     1 |         13
Bốn cột điều kiện khớp nhau với mọi e = 0..12


## 8. Code nộp LeetCode

### Cách chính — bitmask (O(1), không vòng lặp)

In [14]:
class Solution:
    def isPowerOfFour(self, n: int) -> bool:
        return n > 0 and (n & (n - 1)) == 0 and (n & 0x55555555) != 0

print(Solution().isPowerOfFour(16), Solution().isPowerOfFour(5), Solution().isPowerOfFour(1))

True False True


### Cách nộp thay thế — modulo 3

In [15]:
class SolutionMod3:
    def isPowerOfFour(self, n: int) -> bool:
        return n > 0 and (n & (n - 1)) == 0 and n % 3 == 1

print(SolutionMod3().isPowerOfFour(16), SolutionMod3().isPowerOfFour(8))

True False


### Cách nộp dễ hiểu — chia lặp

In [16]:
class SolutionLoop:
    def isPowerOfFour(self, n: int) -> bool:
        if n <= 0:
            return False
        while n % 4 == 0:
            n //= 4
        return n == 1

print(SolutionLoop().isPowerOfFour(16), SolutionLoop().isPowerOfFour(0))

True False


## 9. Vì sao đúng?

**Chiều ⇒:** nếu `n = 4^k = 2^(2k)` thì `n > 0`, có một bit 1 (nên `n & (n-1) = 0`), bit đó ở vị trí `2k` chẵn → trùng một bit của `0x55555555` (vì `2k <= 30`). Và `n % 3 = 1`.

**Chiều ⇐:** `n > 0` và `n & (n - 1) == 0` ⇒ `n = 2^e` với `0 <= e <= 30`. Điều kiện mask khác 0 ⇒ bit thứ `e` của mask là 1 ⇒ `e` chẵn ⇒ `n = 2^(2t) = 4^t`. (Tương tự, `2^e % 3 == 1` ⇔ `e` chẵn.)

**Vì sao `n & (n-1) == 0` ⇔ một bit 1 (với n > 0):** `n - 1` lật bit 1 thấp nhất và mọi bit 0 bên phải nó; các bit cao hơn giữ nguyên. AND ra 0 ⇔ không còn bit 1 nào cao hơn ⇔ chỉ có một bit 1.

**Cách chia lặp:** invariant "n gốc là lũy thừa của 4 ⇔ n hiện tại là lũy thừa của 4", giống bài 326.

## 10. Time / Space Complexity và so sánh

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Chia lặp | `O(log₄ n)` (≤ 15 vòng) | `O(1)` | dễ hiểu |
| Đệ quy | `O(log₄ n)` | `O(log₄ n)` stack | |
| "Ước của 4^15" | `O(1)` | `O(1)` | **SAI** (4 không nguyên tố) |
| **Bitmask** | **`O(1)`** | **`O(1)`** | follow-up, chuẩn phỏng vấn |
| Modulo 3 | `O(1)` | `O(1)` | đẹp về toán |
| `bit_length()` | `O(1)` | `O(1)` | đặc thù Python |

## 11. Bộ test — mẫu đề, biên, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề: 16", sol.isPowerOfFour(16), True)
check("đề: 5", sol.isPowerOfFour(5), False)
check("đề: 1", sol.isPowerOfFour(1), True)
check("0", sol.isPowerOfFour(0), False)
check("âm -4", sol.isPowerOfFour(-4), False)
check("min int", sol.isPowerOfFour(-2**31), False)
check("lũy thừa lẻ của 2: 2", sol.isPowerOfFour(2), False)
check("lũy thừa lẻ của 2: 8", sol.isPowerOfFour(8), False)
check("4^15", sol.isPowerOfFour(4 ** 15), True)
check("2^31 - 1", sol.isPowerOfFour(2**31 - 1), False)
check("2^30 + 4 (hai bit 1)", sol.isPowerOfFour(2**30 + 4), False)

PASS đề: 16: True
PASS đề: 5: False
PASS đề: 1: True
PASS 0: False
PASS âm -4: False
PASS min int: False
PASS lũy thừa lẻ của 2: 2: False
PASS lũy thừa lẻ của 2: 8: False
PASS 4^15: True
PASS 2^31 - 1: False
PASS 2^30 + 4 (hai bit 1): False


In [18]:
import random
random.seed(342)
truth = set(powers4)
methods = [is_power_of_four_loop, is_power_of_four_recursive, is_power_of_four_mask, is_power_of_four_mod3,
           is_power_of_four_bitlen, Solution().isPowerOfFour, SolutionMod3().isPowerOfFour, SolutionLoop().isPowerOfFour]
samples = list(range(-100, 5000)) + [2 ** e for e in range(31)] + [2 ** e + 1 for e in range(31)] + [-(2 ** e) for e in range(32)]
samples += [random.randint(-2**31, 2**31 - 1) for _ in range(3000)]
for n in samples:
    for f in methods:
        assert f(n) == (n in truth), (f, n)
print(f"{len(samples)} giá trị × {len(methods)} cách: tất cả khớp")

8194 giá trị × 8 cách: tất cả khớp


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra lũy thừa của 2
`n > 0 and n & (n - 1) == 0` → `2, 8, 32` bị coi là lũy thừa của 4.

### Lỗi 2: Quên `n > 0`
- `0 & (0 - 1) == 0` → 0 lọt qua điều kiện "lũy thừa của 2".
- Với số âm, số nguyên Python **không giới hạn bit** nên `n & (n - 1)` của số âm khác 0 — Python may mắn loại được. Nhưng ở Java/C++ (int 32-bit), `-2^31 - 1` bị **tràn** thành `2^31 - 1`, và `-2^31 & (2^31 - 1) == 0` → `-2^31` lọt qua! Ta mô phỏng int 32-bit bằng hàm `to_int32`.
- Thói quen tốt cho mọi ngôn ngữ: **luôn viết `n > 0` đầu tiên**.

In [19]:
def only_power_of_two(n):
    return n > 0 and (n & (n - 1)) == 0          # SAI: thiếu điều kiện vị trí chẵn

def to_int32(x: int) -> int:
    # Ép về số nguyên có dấu 32-bit (giống int của Java/C++, có tràn số)
    return ((x & 0xFFFFFFFF) ^ 0x80000000) - 0x80000000

print("chỉ lũy thừa 2:", [n for n in range(1, 40) if only_power_of_two(n)])
print("0 & (0 - 1) =", 0 & (0 - 1), "-> thiếu n > 0 thì 0 qua điều kiện lũy thừa của 2")
x = -2**31
print("Python  : -2^31 & (-2^31 - 1) =", x & (x - 1))
x_minus_1 = to_int32(x - 1)
print("32-bit  : -2^31 - 1 tràn thành", x_minus_1, "| -2^31 & (đó) =", to_int32(x & x_minus_1), "-> lọt qua nếu thiếu n > 0")

chỉ lũy thừa 2: [1, 2, 4, 8, 16, 32]
0 & (0 - 1) = 0 -> thiếu n > 0 thì 0 qua điều kiện lũy thừa của 2
Python  : -2^31 & (-2^31 - 1) = -4294967296
32-bit  : -2^31 - 1 tràn thành 2147483647 | -2^31 & (đó) = 0 -> lọt qua nếu thiếu n > 0


### Lỗi 3: Dùng mask sai `0xAAAAAAAA`
`0xA = 1010₂` → bit ở vị trí **lẻ**. Đây là mask cho "lũy thừa lẻ của 2", ngược hoàn toàn.

### Lỗi 4: Dùng mẹo ước số như bài 326
Đã thấy ở mục 4: `4^15 % 8 == 0`.

### Lỗi 5: `math.log(n, 4).is_integer()` — sai số float
Trong miền 32-bit có thể "tình cờ" đúng, nhưng float không đảm bảo (bài 326 đã sai với `243`). Nếu dùng log, phải kiểm tra lại bằng số nguyên.

In [20]:
import math
wrong_mask = lambda n: n > 0 and (n & (n - 1)) == 0 and (n & 0xAAAAAAAA) != 0
print("mask 0xAAAAAAAA chọn:", [n for n in range(1, 300) if wrong_mask(n)])
print("log trong miền 32-bit:", [p for p in powers4 if not math.log(p, 4).is_integer()], "(may mắn không sai)")
bad_k = [k for k in range(1, 40) if not math.log(4 ** k, 4).is_integer()]
print("nhưng với số mũ lớn hơn, log sai ở k =", bad_k, "-> không đáng tin")
print("log + kiểm tra lại bằng số nguyên:", all(4 ** round(math.log(4 ** k, 4)) == 4 ** k for k in range(60)))

mask 0xAAAAAAAA chọn: [2, 8, 32, 128]
log trong miền 32-bit: [] (may mắn không sai)
nhưng với số mũ lớn hơn, log sai ở k = [29, 31, 39] -> không đáng tin
log + kiểm tra lại bằng số nguyên: True


## 13. Mẹo nhận dạng pattern

- "Lũy thừa của 2" → `n > 0 and n & (n - 1) == 0`.
- "Lũy thừa của 4, 16, …" (lũy thừa của 2 có số mũ chia hết cho m) → lũy thừa của 2 **+** điều kiện vị trí bit (mask hoặc `bit_length`).
- "Lũy thừa của số nguyên tố p" → `MAX_POW_P % n == 0`.
- Mặt nạ hay gặp: `0x55555555` (vị trí chẵn), `0xAAAAAAAA` (vị trí lẻ), `0x33333333`, `0x0F0F0F0F` (dùng trong popcount song song).
- Luôn nhớ `n <= 0` trước mọi mẹo bit.

## 14. Biến thể

### Biến thể 1 — Power of Two (LeetCode 231)
### Biến thể 2 — lũy thừa của 16 (= `2^(4k)`): vị trí bit chia hết cho 4 → mask `0x11111111`
### Biến thể 3 — trả về `log₄ n` (số mũ) hoặc -1, không vòng lặp

In [21]:
def is_power_of_two(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0

def is_power_of_sixteen(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and (n & 0x11111111) != 0

def log4_exact(n: int) -> int:
    if not is_power_of_four_mask(n):
        return -1
    return (n.bit_length() - 1) // 2

print([n for n in range(1, 70000) if is_power_of_sixteen(n)])
print(log4_exact(1), log4_exact(64), log4_exact(32))
print(f"{0x11111111:032b}")

[1, 16, 256, 4096, 65536]
0 3 -1
00010001000100010001000100010001


## 15. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Với `n = 32` và `n = 256`: viết nhị phân, tính `n & (n-1)`, `n & 0x55555555`, `n % 3`. Kết luận?

### Bài 2 — Dự đoán output
`0x5`, `0xF`, `hex(64)`, `(64).bit_length()`, `20 & 19`.

### Bài 3 — Điền chỗ trống

In [22]:
def exercise_power_of_four(n: int) -> bool:
    # TODO 1: n > 0
    # TODO 2: chỉ có 1 bit 1
    # TODO 3: bit đó ở vị trí chẵn (mask hoặc % 3)
    pass

# assert exercise_power_of_four(16) is True
# assert exercise_power_of_four(8) is False
# assert exercise_power_of_four(0) is False

### Bài 4 — Tự viết: lũy thừa của 2 bằng chia lặp (không dùng bit)

In [23]:
def exercise_power_of_two_loop(n: int) -> bool:
    # TODO
    pass

# assert exercise_power_of_two_loop(1) and exercise_power_of_two_loop(1024)
# assert not exercise_power_of_two_loop(0) and not exercise_power_of_two_loop(6)

### Bài 5 — Biến thể: lũy thừa của 8 không vòng lặp
`8^k = 2^(3k)`: vị trí bit chia hết cho 3. Gợi ý: dùng `bit_length()`, hoặc tự xây mask bằng code.

In [24]:
def exercise_power_of_eight(n: int) -> bool:
    # TODO
    pass

# assert [n for n in range(1, 5000) if exercise_power_of_eight(n)] == [1, 8, 64, 512, 4096]

### Bài 6 — Mini challenge: tạo mask tổng quát
Viết `exercise_build_mask(step, bits=32)` trả về số có bit 1 ở các vị trí `0, step, 2·step, …` < `bits`. Kiểm tra `exercise_build_mask(2) == 0x55555555`, `exercise_build_mask(4) == 0x11111111`.

In [25]:
def exercise_build_mask(step: int, bits: int = 32) -> int:
    # TODO: dùng 1 << pos và phép |
    pass

# assert exercise_build_mask(2) == 0x55555555
# assert exercise_build_mask(4) == 0x11111111
# assert exercise_build_mask(3) == 0x49249249

## 16. Gợi ý 3 mức

**Bài 1** — (1) `32 = 100000₂`, `256 = 100000000₂`. (2) Vị trí bit 1 là 5 và 8. (3) Lẻ → False, chẵn → True.

**Bài 2** — (1) Hex → thập phân. (2) `64 = 1000000₂` có 7 bit. (3) `20 = 10100₂`, `19 = 10011₂`.

**Bài 3** — (1) Ba điều kiện nối bằng `and`. (2) Ngoặc quanh phép `&`. (3) Xem mục 6.

**Bài 4** — (1) Như bài 326 nhưng chia 2. (2) Chặn `n <= 0`. (3) `while n % 2 == 0: n //= 2`.

**Bài 5** — (1) Lũy thừa của 2 trước. (2) Vị trí bit = `bit_length() - 1`. (3) Vị trí `% 3 == 0`.

**Bài 6** — (1) Khởi tạo `mask = 0`. (2) `for pos in range(0, bits, step)`. (3) `mask |= 1 << pos`.

## 17. Đáp án tham khảo

**Bài 1:** `32`: `32 & 31 = 0`, `32 & mask = 0`, `32 % 3 = 2` → False. `256`: `256 & 255 = 0`, `256 & mask = 256`, `256 % 3 = 1` → True.

**Bài 2:** `5`, `15`, `'0x40'`, `7`, `16`.

In [26]:
def answer_power_of_four(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and (n & 0x55555555) != 0

def answer_power_of_two_loop(n: int) -> bool:
    if n <= 0:
        return False
    while n % 2 == 0:
        n //= 2
    return n == 1

def answer_power_of_eight(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0 and (n.bit_length() - 1) % 3 == 0

def answer_build_mask(step: int, bits: int = 32) -> int:
    mask = 0
    for pos in range(0, bits, step):
        mask |= 1 << pos
    return mask

assert (32 & 31, 32 & 0x55555555, 32 % 3) == (0, 0, 2) and (256 & 255, 256 & 0x55555555, 256 % 3) == (0, 256, 1)
assert (0x5, 0xF, hex(64), (64).bit_length(), 20 & 19) == (5, 15, "0x40", 7, 16)
assert answer_power_of_four(16) and not answer_power_of_four(8) and not answer_power_of_four(0)
assert answer_power_of_two_loop(1) and answer_power_of_two_loop(1024)
assert not answer_power_of_two_loop(0) and not answer_power_of_two_loop(6)
assert [n for n in range(1, 5000) if answer_power_of_eight(n)] == [1, 8, 64, 512, 4096]
assert answer_build_mask(2) == 0x55555555 and answer_build_mask(4) == 0x11111111 and answer_build_mask(3) == 0x49249249
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isPowerOfFour(self, n: int) -> bool:
        return n > 0 and (n & (n - 1)) == 0 and (n & 0x55555555) != 0
        # hoặc: n > 0 and (n & (n - 1)) == 0 and n % 3 == 1
```

**Template "lũy thừa của 2^m":** `n > 0` → `n & (n-1) == 0` → vị trí bit `(n.bit_length() - 1) % m == 0` (hoặc mask tương ứng).

**Checklist trước khi submit:**
- [ ] Có `n > 0` (loại 0 và số âm)?
- [ ] Có loại lũy thừa **lẻ** của 2 (2, 8, 32, …)?
- [ ] Mask đúng `0x55555555` (vị trí chẵn), không phải `0xAAAAAAAA`?
- [ ] Ngoặc quanh các phép `&`?
- [ ] Đã thử `1, 2, 4, 8, 16, 0, -4`?